# ROOT TH2 efficiency and background maps

This notebook demonstrates ordinary-Dalitz TH2 maps. The ROOT loaders return fitter-ready efficiency/background objects that can be passed directly to `generate_toy` and `FitSession`.


In [ ]:
import numpy as np
import uproot
import matplotlib.pyplot as plt
from dalitzplotfitter import (
    BackgroundSpec,DecayChannel,DecayModel,FitSession,NonResonant,Parameter,RealImag,
    ToyBackground,enable_x64,generate_toy,histogram_background_from_root,
    histogram_efficiency_from_root,plot_dalitz,
)
enable_x64()


In [ ]:
model=DecayModel(
    DecayChannel("B+",("K+","pi+","pi-")),
    [NonResonant(RealImag(1.0,0.0))],
    normalization_method="square-dalitz",normalization_resolution=140,normalization_pair=(0,2),
)
x=np.linspace(0.3,27.0,31)
y=np.linspace(0.08,23.0,31)
xc=0.5*(x[:-1]+x[1:]); yc=0.5*(y[:-1]+y[1:])
X,Y=np.meshgrid(xc,yc,indexing="ij")
eff_values=0.45+0.40*(X-X.min())/(X.max()-X.min())
bkg_values=0.25+0.75*(Y-Y.min())/(Y.max()-Y.min())

with uproot.recreate("maps_dp.root") as f:
    f["efficiency_s13_s23"]=(eff_values,x,y)
    f["background_s13_s23"]=(bkg_values,x,y)

eff=histogram_efficiency_from_root(
    "maps_dp.root","efficiency_s13_s23",x_variable="s13",y_variable="s23"
)
bkg=histogram_background_from_root(
    "maps_dp.root","background_s13_s23",x_variable="s13",y_variable="s23"
)


In [ ]:
f_sig=Parameter("signal_fraction",0.75,bounds=(0.05,0.99))
data=generate_toy(
    model,20_000,efficiency=eff,signal_fraction=0.80,
    backgrounds=(ToyBackground("comb",bkg),),
    seed=1414,pool_size=150_000,
)
plot_dalitz(data,x="s13",y="s23",title="ROOT TH2 efficiency + background toy")
plt.show()

session=FitSession(
    model,data,efficiency=eff,signal_fraction=f_sig,
    backgrounds=(BackgroundSpec("comb",bkg),),
)
result=session.fit()
session.report(result)
session.plot_projection(result,"s13")
plt.show()
